# Evaluate Exp05 and Exp06 on new June 17 bacterial data

This notebook compares only the Exp05 multimodal model and Exp06 robust multimodal model on the newly parsed Bacillus/Micrococcus data. It intentionally ignores helper artifacts such as `label_encoder.joblib` and scaler files.

Expected labels are mapped as:

- `bacillus_cereus_possible` → `B_cereus`
- `micrococcus` → `M_luteus`

Outputs are saved under `results/live_rapid_e/exp05_exp06_new_data_evaluation`.

In [29]:
from __future__ import annotations

import importlib
import inspect
import json
import math
import warnings
from pathlib import Path
from typing import Any, Callable

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cpu


In [30]:
PROJECT_ROOT = Path(
    r"C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis"
)

MODEL_DIRS = {
    "exp05": PROJECT_ROOT / "models" / "trained" / "exp05_multimodal_species_v1",
    "exp06": PROJECT_ROOT / "models" / "trained" / "exp06_robust_multimodal_species_v1",
}

PARSED_DIR = PROJECT_ROOT / "data" / "live_rapid_e" / "parsed"
RESULTS_DIR = PROJECT_ROOT / "results" / "live_rapid_e" / "exp05_exp06_new_data_evaluation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PARSED_DIR = PROJECT_ROOT / r"data\live_rapid_e\parsed"
RESULTS_DIR = PROJECT_ROOT / r"results\live_rapid_e\exp05_exp06_new_data_evaluation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Preferred combined file from the parser created earlier.
COMBINED_PARQUET = PARSED_DIR / "june17_bacillus_cereus_possible_micrococcus_preprocessed.parquet"

# Fallback individual files.
INDIVIDUAL_PARQUETS = [
    PARSED_DIR / "june17_bacillus_cereus_possible_preprocessed.parquet",
    PARSED_DIR / "june17_micrococcus_preprocessed.parquet",
]

print("Project root:", PROJECT_ROOT)
print("Model directories:")
for name, path in MODEL_DIRS.items():
    print(f"  {name}: {path}")

print("Parsed dir:", PARSED_DIR)
print("Results dir:", RESULTS_DIR)


Project root: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis
Model directories:
  exp05: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\models\trained\exp05_multimodal_species_v1
  exp06: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\models\trained\exp06_robust_multimodal_species_v1
Parsed dir: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\data\live_rapid_e\parsed
Results dir: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp05_exp06_new_data_evaluation


In [31]:
def load_model_bundle(model_dir: Path) -> dict:
    bundle = {
        "model_dir": model_dir,
        "model_path": model_dir / "model.pt",
        "spectra_scaler": joblib.load(model_dir / "spectrometer_scaler.joblib"),
        "lifetime_scaler": joblib.load(model_dir / "lifetime_scaler.joblib"),
        "scattering_scaler": joblib.load(model_dir / "scattering_scaler.joblib"),
        "scalar_scaler": joblib.load(model_dir / "scalar_scaler.joblib"),
    }

    checkpoint = torch.load(bundle["model_path"], map_location="cpu", weights_only=False)

    bundle["checkpoint"] = checkpoint
    bundle["class_names"] = checkpoint.get(
        "class_names",
        ["B_cereus", "B_endophyticus", "K_salsicia", "M_luteus", "S_huminis"],
    )

    return bundle

def normalize_label(value: Any) -> str | None:
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None

    text = str(value).strip()

    mapping = {
        "B. cereus": "B_cereus",
        "B cereus": "B_cereus",
        "B_cereus": "B_cereus",
        "Bacillus cereus": "B_cereus",
        "bacillus_cereus_possible": "B_cereus",
        "B_cereus_possible": "B_cereus",
        "bacillus": "B_cereus",

        "M. luteus": "M_luteus",
        "M luteus": "M_luteus",
        "M_luteus": "M_luteus",
        "Micrococcus luteus": "M_luteus",
        "micrococcus": "M_luteus",

        "B. endophyticus": "B_endophyticus",
        "B_endophyticus": "B_endophyticus",
        "Bacillus endophyticus": "B_endophyticus",

        "K. salsicia": "K_salsicia",
        "K_salsicia": "K_salsicia",
        "Kocuria salsicia": "K_salsicia",

        "S. huminis": "S_huminis",
        "S_huminis": "S_huminis",
        "Staphylococcus huminis": "S_huminis",

        "unknown": "unknown",
        "Unknown": "unknown",
    }

    return mapping.get(text, text)


def load_new_data() -> pd.DataFrame:
    if COMBINED_PARQUET.exists():
        files = [COMBINED_PARQUET]
    else:
        files = [p for p in INDIVIDUAL_PARQUETS if p.exists()]

    if not files:
        candidates = sorted(PARSED_DIR.glob("*june17*.parquet"))
        files = [p for p in candidates if "bacillus" in p.name.lower() or "micrococcus" in p.name.lower()]

    if not files:
        raise FileNotFoundError(f"No June 17 Bacillus/Micrococcus parsed parquet files found in {PARSED_DIR}")

    print("Loading:")
    for p in files:
        print(" -", p)

    data = pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)

    if "expected_sample" in data.columns:
        data["y_true"] = data["expected_sample"].map(normalize_label)
    elif "experiment_name" in data.columns:
        data["y_true"] = data["experiment_name"].map(normalize_label)
    else:
        raise ValueError("Could not find expected_sample or experiment_name to create y_true.")

    # Keep only the two target sample types for this comparison.
    data = data[data["y_true"].isin(["B_cereus", "M_luteus"])].reset_index(drop=True)

    return data


df = load_new_data()
print("Rows:", len(df))
print("Columns:", len(df.columns))
print(df["y_true"].value_counts(dropna=False))
df.head()


Loading:
 - C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\data\live_rapid_e\parsed\june17_bacillus_cereus_possible_micrococcus_preprocessed.parquet
Rows: 53
Columns: 1978
y_true
B_cereus    28
M_luteus    25
Name: count, dtype: int64


,raw_file,raw_path,particle_index,timestamp,serial,version,number_of_modules,has_fluorescence,lt11_framelength,lt03_framelength,...,si_1434,si_1435,si_1436,si_1437,si_1438,si_1439,experiment_date,expected_sample,expected_composition,y_true
0,D_000000155_202606171224.raw,C:\Users\chris\OneDrive\Documents\Universitat ...,289,2026-06-17 14:24:10.359,3532891,2,3,True,960,256,...,0.0,0.0,0.0,0.0,0.000000,0.0,2026-06-17,B_cereus,B_cereus_possible,B_cereus
1,D_000000156_202606171225.raw,C:\Users\chris\OneDrive\Documents\Universitat ...,265,2026-06-17 14:25:13.295,3532891,2,3,True,936,256,...,0.0,0.0,0.0,0.0,0.000000,0.0,2026-06-17,B_cereus,B_cereus_possible,B_cereus
2,D_000000156_202606171225.raw,C:\Users\chris\OneDrive\Documents\Universitat ...,1156,2026-06-17 14:25:44.207,3532891,2,3,True,13680,256,...,0.0,0.0,0.0,0.0,0.000000,0.0,2026-06-17,B_cereus,B_cereus_possible,B_cereus
3,D_000000156_202606171225.raw,C:\Users\chris\OneDrive\Documents\Universitat ...,2387,2026-06-17 14:25:59.660,3532891,2,3,True,4296,256,...,0.0,0.0,0.0,0.0,0.011194,0.0,2026-06-17,B_cereus,B_cereus_possible,B_cereus
4,D_000000158_202606171227.raw,C:\Users\chris\OneDrive\Documents\Universitat ...,2115,2026-06-17 14:27:26.556,3532891,2,3,True,912,256,...,0.0,0.0,0.0,0.0,0.000000,0.0,2026-06-17,B_cereus,B_cereus_possible,B_cereus


In [32]:
def sorted_feature_cols(data: pd.DataFrame, prefix: str) -> list[str]:
    # Numeric suffix sort: fs_2 before fs_10.
    def key(c: str):
        tail = c.replace(prefix, "")
        return int(tail) if tail.isdigit() else tail

    return sorted([c for c in data.columns if c.startswith(prefix)], key=key)

FS_COLS = sorted_feature_cols(df, "fs_")
LT_COLS = sorted_feature_cols(df, "lt_")
SI_COLS = sorted_feature_cols(df, "si_")
SCALAR_COLS = [c for c in ["size", "time_asymmetry"] if c in df.columns]

print("fs columns:", len(FS_COLS))
print("lt columns:", len(LT_COLS))
print("si columns:", len(SI_COLS))
print("scalar columns:", SCALAR_COLS)

assert FS_COLS, "No fs_* columns found. Re-run preprocessing with flatten=True."
assert LT_COLS, "No lt_* columns found. Re-run preprocessing with flatten=True."
assert SI_COLS, "No si_* columns found. Re-run preprocessing with flatten=True."


fs columns: 256
lt columns: 256
si columns: 1440
scalar columns: ['size', 'time_asymmetry']


In [33]:
def find_checkpoint(keywords: list[str]) -> Path:
    """Find one .pt/.pth checkpoint whose path contains all keywords."""
    candidates = []
    for path in TRAINED_DIR.rglob("*"):
        if path.suffix.lower() not in {".pt", ".pth"}:
            continue
        text = str(path).lower().replace("\\", "/")
        if all(k.lower() in text for k in keywords):
            candidates.append(path)

    if not candidates:
        raise FileNotFoundError(f"No checkpoint found for keywords={keywords} under {TRAINED_DIR}")

    # Prefer files named model.pt, otherwise shortest path.
    candidates = sorted(candidates, key=lambda p: (p.name != "model.pt", len(str(p))))
    return candidates[0]

# Adjust these manually if your filenames differ.
EXP05_CHECKPOINT = find_checkpoint(["exp05"])
EXP06_CHECKPOINT = find_checkpoint(["exp06"])

print("Exp05 checkpoint:", EXP05_CHECKPOINT)
print("Exp06 checkpoint:", EXP06_CHECKPOINT)


Exp05 checkpoint: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\models\trained\exp05_multimodal_species_v1\model.pt
Exp06 checkpoint: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\models\trained\exp06_robust_multimodal_species_v1\model.pt


In [34]:
DEFAULT_CLASS_NAMES = ["B_cereus", "B_endophyticus", "K_salsicia", "M_luteus", "S_huminis"]


def load_checkpoint(path: Path) -> dict[str, Any]:
    ckpt = torch.load(path, map_location="cpu")
    if isinstance(ckpt, nn.Module):
        return {"full_model": ckpt}
    if not isinstance(ckpt, dict):
        raise TypeError(f"Unsupported checkpoint type for {path}: {type(ckpt)}")
    return ckpt


def class_names_from_checkpoint(ckpt: dict[str, Any]) -> list[str]:
    names = ckpt.get("class_names") or ckpt.get("classes") or ckpt.get("labels")
    if names is None:
        names = DEFAULT_CLASS_NAMES
    return [normalize_label(x) for x in names]


def load_scaler(search_dir: Path, possible_names: list[str]):
    for directory in [search_dir, search_dir.parent, TRAINED_DIR]:
        for name in possible_names:
            path = directory / name
            if path.exists():
                return joblib.load(path), path
    return None, None


def transform_or_identity(x: np.ndarray, scaler):
    if scaler is None:
        return x.astype(np.float32)
    return scaler.transform(x).astype(np.float32)


def build_inputs(data: pd.DataFrame, model_dir: Path) -> dict[str, torch.Tensor]:
    spectra = data[FS_COLS].to_numpy(dtype=np.float32)
    lifetime = data[LT_COLS].to_numpy(dtype=np.float32)
    scattering = data[SI_COLS].to_numpy(dtype=np.float32)
    scalar = data[SCALAR_COLS].to_numpy(dtype=np.float32) if SCALAR_COLS else np.zeros((len(data), 0), dtype=np.float32)

    spectra_scaler, spectra_scaler_path = load_scaler(model_dir, ["spectrometer_scaler.joblib", "spectra_scaler.joblib"])
    lifetime_scaler, lifetime_scaler_path = load_scaler(model_dir, ["lifetime_scaler.joblib"])
    scattering_scaler, scattering_scaler_path = load_scaler(model_dir, ["scattering_scaler.joblib"])
    scalar_scaler, scalar_scaler_path = load_scaler(model_dir, ["scalar_scaler.joblib"])

    if spectra_scaler_path: print("  spectra scaler:", spectra_scaler_path.name)
    if lifetime_scaler_path: print("  lifetime scaler:", lifetime_scaler_path.name)
    if scattering_scaler_path: print("  scattering scaler:", scattering_scaler_path.name)
    if scalar_scaler_path: print("  scalar scaler:", scalar_scaler_path.name)

    spectra = transform_or_identity(spectra, spectra_scaler)
    lifetime = transform_or_identity(lifetime, lifetime_scaler)
    scattering = transform_or_identity(scattering, scattering_scaler)
    scalar = transform_or_identity(scalar, scalar_scaler) if scalar.shape[1] else scalar

    # Common shapes used by the multimodal models.
    spectra_2d = spectra.reshape(len(data), 8, 32)
    lifetime_2d = lifetime.reshape(len(data), 4, 64)
    scattering_2d = scattering.reshape(len(data), 24, -1)
    all_flat = np.concatenate([spectra, lifetime, scattering, scalar], axis=1)

    return {
        "spectra_flat": torch.tensor(spectra, dtype=torch.float32),
        "lifetime_flat": torch.tensor(lifetime, dtype=torch.float32),
        "scattering_flat": torch.tensor(scattering, dtype=torch.float32),
        "scalar": torch.tensor(scalar, dtype=torch.float32),
        "spectra_2d": torch.tensor(spectra_2d, dtype=torch.float32),
        "lifetime_2d": torch.tensor(lifetime_2d, dtype=torch.float32),
        "scattering_2d": torch.tensor(scattering_2d, dtype=torch.float32),
        "all_flat": torch.tensor(all_flat, dtype=torch.float32),
    }


In [35]:
def import_module_candidates(names: list[str]):
    for name in names:
        try:
            return importlib.import_module(name), name
        except Exception:
            pass
    return None, None


def candidate_model_classes(module) -> list[type[nn.Module]]:
    classes = []
    for _, obj in inspect.getmembers(module, inspect.isclass):
        try:
            if issubclass(obj, nn.Module) and obj is not nn.Module:
                classes.append(obj)
        except Exception:
            pass
    return classes


def instantiate_model_from_module(module, ckpt: dict[str, Any], class_names: list[str], inputs: dict[str, torch.Tensor]) -> nn.Module:
    if "full_model" in ckpt:
        return ckpt["full_model"]

    n_classes = int(ckpt.get("n_classes") or len(class_names))
    input_features = ckpt.get("input_features", {})

    constructor_kwargs = {
        "n_classes": n_classes,
        "num_classes": n_classes,
        "output_dim": n_classes,
        "class_names": class_names,
        "spectra_dim": inputs["spectra_flat"].shape[1],
        "spectrometer_dim": inputs["spectra_flat"].shape[1],
        "spec_dim": inputs["spectra_flat"].shape[1],
        "lifetime_dim": inputs["lifetime_flat"].shape[1],
        "scattering_dim": inputs["scattering_flat"].shape[1],
        "scalar_dim": inputs["scalar"].shape[1],
        "input_dim": inputs["all_flat"].shape[1],
        "input_features": input_features,
    }

    preferred_names = [
        str(ckpt.get("architecture", "")),
        str(ckpt.get("model_name", "")),
        "RobustMultimodalCNN",
        "RobustMultimodalSpeciesCNN",
        "MultimodalSpeciesCNN",
        "MultimodalCNN",
        "MultimodalSpeciesModel",
        "MultimodalDeepClassifier",
    ]

    classes = candidate_model_classes(module)
    classes = sorted(
        classes,
        key=lambda cls: min([preferred_names.index(x) for x in preferred_names if x and x == cls.__name__] or [999]),
    )

    errors = []
    for cls in classes:
        sig = inspect.signature(cls.__init__)
        params = [p for p in sig.parameters if p != "self"]

        # Try signature-filtered kwargs.
        kwargs = {k: v for k, v in constructor_kwargs.items() if k in params}
        try:
            return cls(**kwargs)
        except Exception as exc:
            errors.append((cls.__name__, "kwargs", repr(exc)))

        # Try common simple constructors.
        for args in [(), (n_classes,), (inputs["all_flat"].shape[1], n_classes)]:
            try:
                return cls(*args)
            except Exception as exc:
                errors.append((cls.__name__, f"args={args}", repr(exc)))

    available = [cls.__name__ for cls in classes]
    raise RuntimeError(
        "Could not instantiate a model class from the experiment module.\n"
        f"Available nn.Module classes: {available}\n"
        f"Recent constructor errors: {errors[:10]}"
    )


def load_state_dict_into_model(model: nn.Module, ckpt: dict[str, Any]) -> nn.Module:
    if "full_model" in ckpt:
        return model

    state = ckpt.get("model_state_dict") or ckpt.get("state_dict")
    if state is None:
        # Last resort: maybe the checkpoint itself is a state dict.
        state = {k: v for k, v in ckpt.items() if torch.is_tensor(v)}

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"  state_dict load: missing={len(missing)}, unexpected={len(unexpected)}")

    # If almost nothing loaded, stop instead of silently evaluating a random model.
    total_keys = len(model.state_dict())
    if len(missing) / max(total_keys, 1) > 0.50:
        raise RuntimeError(
            "More than 50% of model parameters were missing when loading the checkpoint. "
            "This usually means the wrong architecture class was instantiated."
        )

    return model


In [36]:
def forward_model(model: nn.Module, batch: dict[str, torch.Tensor]) -> torch.Tensor:
    """Try common forward signatures used in the Exp05/Exp06 multimodal models."""
    attempts = [
        lambda: model(batch["spectra_2d"], batch["lifetime_2d"], batch["scattering_2d"], batch["scalar"]),
        lambda: model(batch["spectra_flat"], batch["lifetime_flat"], batch["scattering_flat"], batch["scalar"]),
        lambda: model(batch["spectra_2d"], batch["lifetime_2d"], batch["scattering_flat"], batch["scalar"]),
        lambda: model(batch["spectra_flat"], batch["lifetime_flat"], batch["scattering_2d"], batch["scalar"]),
        lambda: model({
            "spectra": batch["spectra_2d"],
            "spectrometer": batch["spectra_2d"],
            "lifetime": batch["lifetime_2d"],
            "scattering": batch["scattering_2d"],
            "scalar": batch["scalar"],
        }),
        lambda: model(batch["all_flat"]),
    ]

    last_error = None
    for attempt in attempts:
        try:
            out = attempt()
            if isinstance(out, tuple):
                out = out[0]
            if isinstance(out, dict):
                out = out.get("logits") or out.get("output") or out.get("pred")
            if out is not None:
                return out
        except Exception as exc:
            last_error = exc

    raise RuntimeError(f"No compatible forward signature worked. Last error: {repr(last_error)}")


def predict_with_model(model: nn.Module, inputs: dict[str, torch.Tensor], class_names: list[str], batch_size: int = 512):
    model = model.to(DEVICE)
    model.eval()
    n = inputs["all_flat"].shape[0]

    pred_idx_all = []
    proba_all = []

    with torch.no_grad():
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            batch = {k: v[start:end].to(DEVICE) for k, v in inputs.items()}
            logits = forward_model(model, batch)

            if logits.ndim == 1:
                logits = logits.unsqueeze(0)

            proba = torch.softmax(logits, dim=1)
            pred_idx = torch.argmax(proba, dim=1)

            pred_idx_all.append(pred_idx.cpu().numpy())
            proba_all.append(proba.cpu().numpy())

    pred_idx = np.concatenate(pred_idx_all)
    proba = np.concatenate(proba_all)
    labels = np.array([class_names[int(i)] if int(i) < len(class_names) else str(int(i)) for i in pred_idx])
    return labels, proba


def apply_exp06_rejection_if_available(pred_labels: np.ndarray, proba: np.ndarray, ckpt: dict[str, Any], inputs: dict[str, torch.Tensor]) -> np.ndarray:
    """Apply simple checkpoint rejection thresholds when present. This mirrors Exp06 conservatively."""
    rejected = np.zeros(len(pred_labels), dtype=bool)
    cfg = ckpt.get("rejection_config") or {}

    if hasattr(cfg, "__dict__"):
        cfg = cfg.__dict__

    softmax_threshold = cfg.get("softmax_threshold", None)
    margin_threshold = cfg.get("margin_threshold", None)

    max_proba = proba.max(axis=1)
    sorted_proba = np.sort(proba, axis=1)
    margin = sorted_proba[:, -1] - sorted_proba[:, -2] if proba.shape[1] > 1 else max_proba

    if softmax_threshold is not None:
        rejected |= max_proba < float(softmax_threshold)
    if margin_threshold is not None:
        rejected |= margin < float(margin_threshold)

    robust = pred_labels.copy()
    robust[rejected] = "unknown"
    return robust


In [37]:
def evaluate_model_output(model_id: str, y_true: pd.Series, y_pred: np.ndarray, proba: np.ndarray | None = None) -> dict[str, Any]:
    y_true_arr = y_true.map(normalize_label).to_numpy()
    y_pred_arr = np.array([normalize_label(x) for x in y_pred])

    mask = pd.notna(y_true_arr) & pd.notna(y_pred_arr)
    yt = y_true_arr[mask]
    yp = y_pred_arr[mask]

    known_mask = yp != "unknown"

    row = {
        "model_id": model_id,
        "n_particles": int(mask.sum()),
        "unknown_fraction": float(np.mean(yp == "unknown")) if len(yp) else np.nan,
        "accuracy": float(accuracy_score(yt, yp)) if len(yp) else np.nan,
        "balanced_accuracy": float(balanced_accuracy_score(yt, yp)) if len(yp) else np.nan,
        "macro_f1": float(f1_score(yt, yp, average="macro", zero_division=0)) if len(yp) else np.nan,
        "known_only_n": int(known_mask.sum()),
        "known_only_accuracy": float(accuracy_score(yt[known_mask], yp[known_mask])) if known_mask.sum() else np.nan,
        "known_only_balanced_accuracy": float(balanced_accuracy_score(yt[known_mask], yp[known_mask])) if known_mask.sum() else np.nan,
    }

    if proba is not None:
        row["mean_confidence"] = float(proba.max(axis=1).mean())
        row["median_confidence"] = float(np.median(proba.max(axis=1)))

    return row


def prediction_fraction_table(pred_df: pd.DataFrame) -> pd.DataFrame:
    out = (
        pred_df
        .groupby(["model_id", "y_true", "y_pred"])
        .size()
        .reset_index(name="n")
    )
    out["fraction"] = out["n"] / out.groupby(["model_id", "y_true"])["n"].transform("sum")
    return out.sort_values(["model_id", "y_true", "fraction"], ascending=[True, True, False])


In [44]:
import torch
import torch.nn as nn


class MultimodalSpeciesCNN(nn.Module):
    def __init__(
        self,
        n_classes,
        spectra_dim=256,
        lifetime_dim=256,
        scattering_dim=1440,
        scalar_dim=2,
    ):
        super().__init__()

        self.spectra_branch = nn.Sequential(
            nn.Linear(spectra_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 128),
            nn.ReLU(),
        )

        self.lifetime_branch = nn.Sequential(
            nn.Linear(lifetime_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 128),
            nn.ReLU(),
        )

        self.scattering_branch = nn.Sequential(
            nn.Linear(scattering_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(512, 128),
            nn.ReLU(),
        )

        self.scalar_branch = nn.Sequential(
            nn.Linear(scalar_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
        )

        self.classifier = nn.Sequential(
            nn.Linear(400, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, n_classes),
        )

    def forward(self, spectra, lifetime, scattering, scalar):
        z_spec = self.spectra_branch(spectra)
        z_life = self.lifetime_branch(lifetime)
        z_scat = self.scattering_branch(scattering)
        z_scalar = self.scalar_branch(scalar)

        z = torch.cat([z_spec, z_life, z_scat, z_scalar], dim=1)
        return self.classifier(z)


def build_model_from_checkpoint(ckpt):
    class_names = ckpt.get(
        "class_names",
        ["B_cereus", "B_endophyticus", "K_salsicia", "M_luteus", "S_huminis"],
    )

    n_classes = ckpt.get("n_classes", len(class_names))

    return MultimodalSpeciesCNN(n_classes=n_classes)

In [45]:
MODEL_SPECS = {
    "exp05": MODEL_DIRS["exp05"] / "model.pt",
    "exp06": MODEL_DIRS["exp06"] / "model.pt",
}

all_metric_rows = []
all_prediction_frames = []
failures = []

for model_id, ckpt_path in MODEL_SPECS.items():
    print("\n" + "=" * 80)
    print(f"Evaluating {model_id}: {ckpt_path}")

    try:
        ckpt = load_checkpoint(ckpt_path)
        class_names = class_names_from_checkpoint(ckpt)
        print("  class names:", class_names)

        inputs = build_inputs(df, ckpt_path.parent)

        model = build_model_from_checkpoint(ckpt)
        print("  model class:", model.__class__.__name__)

        model = load_state_dict_into_model(model, ckpt)

        closed_pred, proba = predict_with_model(model, inputs, class_names)

        if model_id == "exp06":
            robust_pred = apply_exp06_rejection_if_available(
                closed_pred,
                proba,
                ckpt,
                inputs,
            )
            eval_variants = {
                "exp06_closed_set": closed_pred,
                "exp06_robust": robust_pred,
            }
        else:
            eval_variants = {
                "exp05_closed_set": closed_pred,
            }

        for variant_id, pred in eval_variants.items():
            metrics = evaluate_model_output(
                variant_id,
                df["y_true"],
                pred,
                proba,
            )
            metrics["checkpoint"] = str(ckpt_path)
            all_metric_rows.append(metrics)

            pred_frame = pd.DataFrame({
                "model_id": variant_id,
                "experiment_name": df["experiment_name"] if "experiment_name" in df.columns else None,
                "expected_sample": df["expected_sample"] if "expected_sample" in df.columns else None,
                "y_true": df["y_true"],
                "y_pred": pred,
                "closed_set_pred": closed_pred,
                "confidence": proba.max(axis=1),
            })

            for col in ["raw_file", "particle_index", "source_file"]:
                if col in df.columns:
                    pred_frame[col] = df[col]

            all_prediction_frames.append(pred_frame)

            print(
                f"  {variant_id}: accuracy={metrics['accuracy']:.4f}, "
                f"balanced_accuracy={metrics['balanced_accuracy']:.4f}, "
                f"macro_f1={metrics['macro_f1']:.4f}, "
                f"unknown_fraction={metrics['unknown_fraction']:.4f}"
            )

    except Exception as exc:
        print("  FAILED:", repr(exc))
        failures.append({
            "model_id": model_id,
            "checkpoint": str(ckpt_path),
            "error": repr(exc),
        })

metrics_df = (
    pd.DataFrame(all_metric_rows)
    .sort_values("balanced_accuracy", ascending=False, na_position="last")
    if all_metric_rows
    else pd.DataFrame()
)

predictions_df = (
    pd.concat(all_prediction_frames, ignore_index=True)
    if all_prediction_frames
    else pd.DataFrame()
)

failures_df = pd.DataFrame(failures)

metrics_df


Evaluating exp05: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\models\trained\exp05_multimodal_species_v1\model.pt
  class names: ['B_cereus', 'B_endophyticus', 'K_salsicia', 'M_luteus', 'S_huminis']
  spectra scaler: spectrometer_scaler.joblib
  lifetime scaler: lifetime_scaler.joblib
  scattering scaler: scattering_scaler.joblib
  scalar scaler: scalar_scaler.joblib
  model class: MultimodalSpeciesCNN
  FAILED: RuntimeError('Error(s) in loading state_dict for MultimodalSpeciesCNN:\n\tsize mismatch for classifier.0.weight: copying a param with shape torch.Size([128, 208]) from checkpoint, the shape in current model is torch.Size([256, 400]).\n\tsize mismatch for classifier.0.bias: copying a param with shape torch.Size([128]) from checkpoint, the shape in current model is torch.Size([256]).\n\tsize mismatch for classifier.4.weight: copying a param with shape torch.Size([64, 128]) from checkpoint, the shape in current model is torch.Size([128, 256]).\n\tsize mi

""


In [46]:
metrics_path = RESULTS_DIR / "exp05_exp06_metrics.csv"
predictions_path = RESULTS_DIR / "exp05_exp06_particle_predictions.parquet"
fractions_path = RESULTS_DIR / "exp05_exp06_prediction_fractions.csv"
failures_path = RESULTS_DIR / "exp05_exp06_failures.csv"

metrics_df.to_csv(metrics_path, index=False)

if not predictions_df.empty:
    predictions_df.to_parquet(predictions_path, index=False)
    fractions_df = prediction_fraction_table(predictions_df)
    fractions_df.to_csv(fractions_path, index=False)
else:
    fractions_df = pd.DataFrame()

failures_df.to_csv(failures_path, index=False)

print("Saved metrics:", metrics_path)
print("Saved predictions:", predictions_path if not predictions_df.empty else None)
print("Saved fractions:", fractions_path if not fractions_df.empty else None)
print("Saved failures:", failures_path)

metrics_df


Saved metrics: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp05_exp06_new_data_evaluation\exp05_exp06_metrics.csv
Saved predictions: None
Saved fractions: None
Saved failures: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp05_exp06_new_data_evaluation\exp05_exp06_failures.csv


""


In [47]:
if not fractions_df.empty:
    display(fractions_df)
else:
    print("No prediction fractions available because no model completed successfully.")


No prediction fractions available because no model completed successfully.


In [48]:
# Confusion matrices and classification reports.
for model_id in predictions_df["model_id"].unique() if not predictions_df.empty else []:
    sub = predictions_df[predictions_df["model_id"] == model_id]
    labels = sorted(set(sub["y_true"].dropna()) | set(sub["y_pred"].dropna()))

    print("\n" + "=" * 80)
    print(model_id)
    print(classification_report(sub["y_true"], sub["y_pred"], labels=labels, zero_division=0))

    cm = pd.DataFrame(
        confusion_matrix(sub["y_true"], sub["y_pred"], labels=labels),
        index=[f"true_{x}" for x in labels],
        columns=[f"pred_{x}" for x in labels],
    )
    display(cm)


In [49]:
# Optional visual comparison.
import matplotlib.pyplot as plt

if not metrics_df.empty:
    plot_df = metrics_df.set_index("model_id")[["accuracy", "balanced_accuracy", "macro_f1", "unknown_fraction"]]
    ax = plot_df.plot(kind="bar", figsize=(10, 5))
    ax.set_ylim(0, 1)
    ax.set_title("Exp05 vs Exp06 performance on new June 17 bacterial samples")
    ax.set_ylabel("score / fraction")
    ax.legend(loc="best")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    fig_path = RESULTS_DIR / "exp05_exp06_metric_comparison.png"
    plt.savefig(fig_path, dpi=200)
    print("Saved plot:", fig_path)


In [50]:
def make_brief_summary(metrics: pd.DataFrame, failures: pd.DataFrame) -> str:
    if metrics.empty:
        text = "No Exp05/Exp06 model completed successfully. Check exp05_exp06_failures.csv for the error details."
        return text

    best = metrics.sort_values("balanced_accuracy", ascending=False).iloc[0]
    lines = [
        "# Exp05 vs Exp06 new-data evaluation summary",
        "",
        f"Best variant by balanced accuracy: **{best['model_id']}**.",
        f"- Accuracy: {best['accuracy']:.4f}",
        f"- Balanced accuracy: {best['balanced_accuracy']:.4f}",
        f"- Macro F1: {best['macro_f1']:.4f}",
        f"- Unknown fraction: {best['unknown_fraction']:.4f}",
        "",
        "Interpretation note: this is not a clean held-out validation set from the original training distribution. It is a new live/aerosolized experiment, so low accuracy should be interpreted as distribution shift and/or experimental protocol mismatch, not simply model failure.",
    ]

    if not failures.empty:
        lines.append("")
        lines.append(f"{len(failures)} model(s) failed to run. See `exp05_exp06_failures.csv`.")

    return "\n".join(lines)

summary = make_brief_summary(metrics_df, failures_df)
summary_path = RESULTS_DIR / "exp05_exp06_summary.md"
summary_path.write_text(summary, encoding="utf-8")
print(summary)
print("\nSaved summary:", summary_path)


No Exp05/Exp06 model completed successfully. Check exp05_exp06_failures.csv for the error details.

Saved summary: C:\Users\chris\OneDrive\Documents\Universitat de Barcelona\lif_thesis\results\live_rapid_e\exp05_exp06_new_data_evaluation\exp05_exp06_summary.md
